# Phase 4 — Render–Edit–Refit with VACE (method M3, generalist editor)
Renders the orbit from the original 3DGS, edits it with Wan2.1-VACE-1.3B under two
mask protocols, re-fits a fresh 3DGS on the edits, and scores against clean plates.
Runtime: GPU. After the install cell: restart, re-run cells 1–2, skip install.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Restore dataset + checkpoint; remove Phase-3's edited checkpoint so the
# ORIGINAL scene (object present) is what gets rendered; rebuild splits
import os, glob, shutil, json
DRIVE = '/content/drive/MyDrive/light-footprint-removal'
DATA = '/content/data/footprint'
if not os.path.exists(f'{DATA}/transforms.json'):
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.copytree(f'{DRIVE}/renders/footprint_dataset', DATA)
if not glob.glob('/content/outputs/**/config.yml', recursive=True):
    shutil.rmtree('/content/outputs', ignore_errors=True)
    shutil.copytree(f'{DRIVE}/checkpoints/phase2_with/outputs', '/content/outputs')
for f in glob.glob('/content/outputs/**/step-000015000.ckpt', recursive=True):
    os.remove(f)

root = f'{DATA}/with'
meta = json.load(open(f'{DATA}/transforms.json'))
frames = meta['frames']
test_idx = set(range(0, len(frames), 8))
for name, fr in {'train': [f for i, f in enumerate(frames) if i not in test_idx],
                 'test':  [f for i, f in enumerate(frames) if i in test_idx],
                 'val':   [f for i, f in enumerate(frames) if i in test_idx]}.items():
    json.dump({'camera_angle_x': meta['camera_angle_x'], 'frames': fr},
              open(f'{root}/transforms_{name}.json', 'w'))
print('frames:', len(os.listdir(f'{root}/rgb')))

In [ ]:
# Install. numpy pinned last to 2.0.2 (checkpoint requirement).
# Restart afterwards, re-run cells 1-2, skip this cell.
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
!pip -q install nerfstudio
!pip -q install "diffusers>=0.34" transformers accelerate ftfy imageio imageio-ffmpeg
!pip -q install --force-reinstall "numpy==2.0.2" 

In [ ]:
# Render all 81 views from the original 3DGS (the pipeline's honest input)
import glob, os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
CONFIG = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True),
                key=os.path.getmtime)[-1]
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render dataset --load-config "$CONFIG" \
  --split train --output-path /content/orbit
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render dataset --load-config "$CONFIG" \
  --split test --output-path /content/orbit

In [ ]:
# Merge both splits back into one 81-frame orbit (filenames preserve order)
import shutil
os.makedirs('/content/orbit_frames', exist_ok=True)
for split in ('train', 'test'):
    for f in glob.glob(f'/content/orbit/{split}/rgb/*.jpg'):
        shutil.copy(f, '/content/orbit_frames/' + os.path.basename(f))
frame_paths = sorted(glob.glob('/content/orbit_frames/*.jpg'))
assert len(frame_paths) == 81

In [ ]:
# Editor inputs. Mask polarity for VACE/ROSE: white = generate, black = keep.
# Protocol 1 (object): dilated object silhouettes.
# Protocol 2 (oracle): object + GT-changed pixels; upper bound on localization.
from PIL import Image, ImageFilter
import numpy as np
W, H, NF = 832, 480, 81
video = [Image.open(p).convert('RGB').resize((W, H)) for p in frame_paths]

def binarize(m): return m.point(lambda v: 255 if v > 127 else 0)

mask_obj, mask_fp = [], []
for i in range(1, NF + 1):
    m = Image.open(f'{root}/mask/{i:04d}.png').convert('L').resize((W, H))
    mask_obj.append(binarize(m.filter(ImageFilter.MaxFilter(25))))

    a = np.asarray(Image.open(f'{root}/rgb/{i:04d}.png').convert('RGB'), np.float32) / 255.
    b = np.asarray(Image.open(f'{DATA}/without/rgb/{i:04d}.png').convert('RGB'), np.float32) / 255.
    changed = np.abs(a - b).max(axis=2) > 0.05
    obj = np.array(Image.open(f'{root}/mask/{i:04d}.png').convert('L')) > 127
    m = Image.fromarray(((changed | obj) * 255).astype('uint8')).resize((W, H))
    mask_fp.append(binarize(m.filter(ImageFilter.MaxFilter(15))))
print('editable fraction (oracle):',
      round(float(np.mean([np.array(m).mean() / 255 for m in mask_fp])), 3))

In [ ]:
# Load VACE (1.3B; VAE kept at fp32 to avoid colour banding)
import torch
from diffusers import AutoencoderKLWan, WanVACEPipeline
model_id = 'Wan-AI/Wan2.1-VACE-1.3B-diffusers'
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder='vae', torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16).to('cuda')

In [ ]:
# Prompts describe the TARGET state; negatives name the footprint effects
prompt = ('an empty room with a polished reflective gray floor, a white wall '
          'and a wall mirror; the scene is empty, nothing on the floor; '
          'clean floor, clean mirror reflecting an empty room')
negative = ('red sphere, ball, round object, reflection of a sphere in the '
            'mirror, shadow of an object on the floor, pink glow')

In [ ]:
# Edit under both protocols (seeded for reproducibility)
from diffusers.utils import export_to_video

def edit(mask_list, tag, steps=30, gs=5.0, seed=0):
    out = pipe(video=video, mask=mask_list, prompt=prompt, negative_prompt=negative,
               height=H, width=W, num_frames=NF, num_inference_steps=steps,
               guidance_scale=gs, generator=torch.Generator().manual_seed(seed)).frames[0]
    d = f'/content/edited_{tag}'
    os.makedirs(f'{d}/rgb', exist_ok=True)
    for i, fr in enumerate(out, start=1):
        img = fr if isinstance(fr, Image.Image) else \
              Image.fromarray((np.asarray(fr) * 255).clip(0, 255).astype('uint8'))
        img.resize((W, H)).save(f'{d}/rgb/{i:04d}.png')
    export_to_video(out, f'{d}.mp4', fps=16)
    print('saved', d)

edit(mask_obj, 'run1_objectmask')
edit(mask_fp,  'run2_oraclemask')

In [ ]:
# Optional prompt ablation: emptiness-dominant wording, higher guidance.
# Outcome in our experiments: substitution persists (orange sphere) -> not refit.
# edit(mask_fp, 'run2b_strongprompt', steps=50, gs=7.5, seed=1)

In [ ]:
# Eyeball gate before spending refit time
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, (p, t) in zip(ax, [('/content/orbit_frames/0041.jpg', 'original'),
                          ('/content/edited_run1_objectmask/rgb/0041.png', 'VACE: object mask'),
                          ('/content/edited_run2_oraclemask/rgb/0041.png', 'VACE: oracle mask')]):
    a.imshow(Image.open(p)); a.set_title(t); a.axis('off')
plt.show()

In [ ]:
# Export orbit + mask videos for the ROSE notebooks
from diffusers.utils import export_to_video
A = f'{DRIVE}/rose_inputs'
os.makedirs(A, exist_ok=True)
shutil.copytree('/content/orbit_frames', f'{A}/orbit_frames', dirs_exist_ok=True)
export_to_video(video, f'{A}/orbit.mp4', fps=16)
export_to_video([m.convert('RGB') for m in mask_obj], f'{A}/mask_object.mp4', fps=16)
export_to_video([m.convert('RGB') for m in mask_fp],  f'{A}/mask_oracle.mp4', fps=16)
print('saved to', A)

In [ ]:
# Re-fit a fresh 3DGS on an edited video (poses unchanged -> reuse split files)
def refit(tag, iters=15000):
    ds = f'{DATA}/edited_{tag}'
    os.makedirs(ds, exist_ok=True)
    if not os.path.isdir(f'{ds}/rgb'):
        shutil.copytree(f'/content/edited_{tag}/rgb', f'{ds}/rgb')
    shutil.copy(f'{DATA}/transforms.json', f'{ds}/transforms.json')
    for s in ('train', 'test', 'val'):
        shutil.copy(f'{root}/transforms_{s}.json', f'{ds}/transforms_{s}.json')
    get_ipython().system(
        f'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-train splatfacto '
        f'--data {ds} --output-dir /content/out_{tag} --max-num-iterations {iters} '
        f'--viewer.quit-on-train-completion True --vis tensorboard blender-data')

# Same metric as Phase 3, applied to the re-fitted edited scene
def score(tag):
    cfg = sorted(glob.glob(f'/content/out_{tag}/**/config.yml', recursive=True),
                 key=os.path.getmtime)[-1]
    get_ipython().system(
        f'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render dataset --load-config {cfg} '
        f'--split test --output-path /content/rt_{tag}')
    rendered = sorted(glob.glob(f'/content/rt_{tag}/test/rgb/*.jpg'))
    test_ids = sorted(int(f['file_path'].split('/')[-1])
                      for i, f in enumerate(frames) if i in test_idx)
    def load(p): return np.asarray(Image.open(p).convert('RGB').resize((W, H)),
                                   np.float32) / 255.
    def psnr(a, b, reg=None):
        d = (a - b) ** 2
        if reg is not None:
            if reg.sum() == 0: return float('nan')
            d = d[reg]
        return float(-10 * np.log10(d.mean() + 1e-12))
    full, fp = [], []
    for k, fid in enumerate(test_ids):
        gt_w, gt_c = load(f'{root}/rgb/{fid:04d}.png'), load(f'{DATA}/without/rgb/{fid:04d}.png')
        obj = np.array(Image.open(f'{root}/mask/{fid:04d}.png').resize((W, H))) > 127
        reg = (np.abs(gt_w - gt_c).max(axis=2) > 0.05) & ~obj
        r = load(rendered[k])
        full.append(psnr(r, gt_c)); fp.append(psnr(r, gt_c, reg))
    res = {'tag': tag, 'full': round(float(np.nanmean(full)), 2),
           'footprint': round(float(np.nanmean(fp)), 2)}
    print(res)
    return res

In [ ]:
refit('run1_objectmask'); r1 = score('run1_objectmask')
refit('run2_oraclemask'); r2 = score('run2_oraclemask')

In [ ]:
# After running phase4b (ROSE) and phase5 (SAM2), score those edits here
SRC = f'{DRIVE}/checkpoints/phase4_rose'
results = {}
for tag in ('rose_objectmask', 'rose_oraclemask', 'rose_sam2mask'):
    d = f'/content/edited_{tag}'
    if not os.path.isdir(d) and os.path.isdir(f'{SRC}/edited_{tag}'):
        shutil.copytree(f'{SRC}/edited_{tag}', d)
    if os.path.isdir(d):
        refit(tag)
        results[tag] = score(tag)
print(results)


In [ ]:
# Persist everything
OUT = f'{DRIVE}/checkpoints/phase4_results'
os.makedirs(OUT, exist_ok=True)
for d in glob.glob('/content/edited_*') + glob.glob('/content/out_*') + glob.glob('/content/rt_*'):
    dst = f'{OUT}/{os.path.basename(d)}'
    (shutil.copy if os.path.isfile(d) else
     lambda a, b: shutil.copytree(a, b, dirs_exist_ok=True))(d, dst)
print('saved to', OUT)